# AutoML mit Optuna

**Autor:** jd · **Datum:** 2026-10-28 · **Issue:** –

**Ziel:** Statt vieler Einzel-Experimente eine Suche über **Modellfamilie × Hyperparameter ×
Preprocessing** auf einmal (`awp2.automl.automl`). TPE schlägt den nächsten Versuch anhand der
bisherigen vor; ein Versuch, der nach einem Fold unter dem Median der anderen liegt, wird
abgebrochen (Pruning). Referenz: `combined_svm_aez_month` (CV 0.864, Validierung 0.869).

**Aufbau:** Teil A erzeugt die Läufe (MLflow), Teil B belegt jede Erkenntnis mit einem Plot.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

from awp2.automl import AutoMLConfig, automl
from awp2.config import AUTOML_APPROACH, MODEL_DOCS_FIGURES_DIR
from awp2.data import load_split
from awp2.evaluation import compare_predictions, plot_confusion_matrices
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models.combined import make_combined_svm
from awp2.plots import (
    plot_class_recall,
    plot_cv_scores,
    plot_metric_comparison,
    plot_top_confusions,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

### Referenz: bestes Modell bisher

RBF-SVM auf der kombinierten Klasse mit AEZ × Monat (`combined_svm_aez_month`, Tuning `d93cea89`).

In [ ]:
reference_model = make_combined_svm().set_params(estimator__C=300.0, estimator__gamma=0.003)
reference_config = RunConfig(
    name="combined_svm_aez_month",
    approach="combined",
    preprocessing=PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True),
    track=False,
)
REFERENCE_CV = 0.864

### Suche

Höchstens 300 Versuche oder 1 h. Der Stand liegt in `models/automl.db`: Ein unterbrochener Lauf
setzt mit demselben Namen fort.

In [ ]:
search = automl(
    AutoMLConfig(
        name="automl_optuna",
        description="Optuna (TPE + Median-Pruning) über SVM, Extra Trees, HistGradientBoosting, "
        "logistische Regression und MLP samt Skalierung, SNV, Savitzky-Golay, PCA, Indizes, "
        "AEZ x Monat und Monatskodierung.",
    )
)
print(f"Bester CV-Score: {search.best_score:.4f}")
search.best_params

### Gepaarter Vergleich auf dem Trainingsteil (out-of-fold)

Bei vielen Versuchen ist der beste CV-Score optimistisch – der Sieger ist auch der, der mit den
Folds am meisten Glück hatte. Deshalb Vorhersagen beider Modelle für dieselben Zeilen vergleichen.

In [ ]:
best_config = RunConfig(
    name="automl_optuna_best",
    approach=AUTOML_APPROACH,
    preprocessing=search.best.preprocessing,
    balance_samples=search.best.balance_samples,
    track=False,
)
y_train = load_split().y_train
oof_reference = out_of_fold_predictions(reference_model, reference_config)
oof_best = out_of_fold_predictions(search.best.model, best_config)
compare_predictions(y_train, oof_reference, oof_best)

### Bester Kandidat: einmal auf der Validierung bewerten

In [ ]:
best = run(
    search.best.model,
    RunConfig(
        name="automl_optuna_best",
        approach=AUTOML_APPROACH,
        description="Bester Versuch der Optuna-Suche automl_optuna, einmal auf der Validierung.",
        preprocessing=search.best.preprocessing,
        balance_samples=search.best.balance_samples,
        tuning_run=search.run_id,
    ),
)
reference = run(reference_model, reference_config)
pd.DataFrame({"Referenz-SVM": reference.metrics.model_dump(), "AutoML": best.metrics.model_dump()})

# B · Erkenntnisse

## 1. Suchverlauf

Jeder Punkt ist ein abgeschlossener Versuch, die Linie der beste bisher. Abgebrochene Versuche
(Pruning) fehlen – sie haben nur einen Teil der Folds gerechnet.

In [ ]:
states = pd.Series([trial.state.name for trial in search.study.trials]).value_counts()
print(states.to_string())
ax = plot_optimization_history(search.study)
ax.axhline(REFERENCE_CV, color="black", linestyle="--", linewidth=1, label="Referenz-SVM")
ax.legend()
save_doc_figure(ax.figure, "automl_verlauf", MODEL_DOCS_FIGURES_DIR)

## 2. Bester Versuch je Modellfamilie

In [ ]:
families = search.candidates["params"].map(lambda params: params["family"])
by_family = dict(tuple(search.candidates.groupby(families)))
ax = plot_cv_scores(by_family, reference=REFERENCE_CV)
ax.set_title("Optuna: bester CV-Score je Modellfamilie (gestrichelt: Referenz-SVM)")
save_doc_figure(ax.figure, "automl_familien", MODEL_DOCS_FIGURES_DIR)

## 3. Welche Entscheidungen zählen

Wichtigkeit der Suchparameter (fANOVA über die abgeschlossenen Versuche).

In [ ]:
ax = plot_param_importances(search.study)
save_doc_figure(ax.figure, "automl_wichtigkeit", MODEL_DOCS_FIGURES_DIR)

## 4. Validierung: AutoML gegen die Referenz

In [ ]:
ax = plot_metric_comparison({"Referenz-SVM": reference.metrics, "AutoML (Optuna)": best.metrics})
ax.set_title("AutoML (Optuna) vs. Referenz-SVM")
save_doc_figure(ax.figure, "automl_vergleich", MODEL_DOCS_FIGURES_DIR)

## 5. Fehlerbild des besten Kandidaten

In [ ]:
fig = plot_confusion_matrices(best.y_val, best.y_pred)
save_doc_figure(fig, "automl_confusion", MODEL_DOCS_FIGURES_DIR)

In [ ]:
plot_top_confusions(best.y_val, best.y_pred)

In [ ]:
plot_class_recall(best.y_val, best.y_pred)

# Fazit

| | CV | BAcc Kultur | BAcc Stadium | BAcc kombiniert | Macro-F1 kombiniert |
| --- | --- | --- | --- | --- | --- |
| Referenz-SVM (`combined_svm_aez_month`) | 0.864 | 0.923 | 0.922 | 0.869 | 0.859 |
| AutoML (Optuna), RBF-SVM | 0.867 | 0.936 | 0.916 | **0.879** | 0.853 |

- 542 Versuche in 1 h; ab etwa Versuch 280 ein Plateau bei 0.867 – kein übersehenes Optimum.
- `gamma` und `C` der SVM erklären fast alle Unterschiede, keine Vorverarbeitungsoption zählt
  (Wichtigkeit < 0.01): Die Vorverarbeitung ist ausgereizt.
- CV-Gewinn +0.003 < `MIN_CV_GAIN`; out-of-fold liegt die Referenz sogar vorn (116 : 77,
  p = 0.006). Die +0.010 auf der Validierung zählen deshalb **nicht** als neuer Bestwert.

Details: `docs/modelle/ansaetze/automl.md`.